### Imports

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import sys
from pathlib import Path

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))


from src.paths import *
from src.graph_utils import (
    build_grid_graph,
    calculate_graph_metrics,
    add_solar_distance
)

### Calculate centrality

In [3]:
substations_path = GIS / "grid_substations.csv"
lines_path = GIS / "transmission_lines.csv"
solar_path = GIS / "solar_power_plants.csv"
if all(path.is_file() for path in (substations_path, lines_path, solar_path)):
    substations = pd.read_csv(substations_path)
    lines = pd.read_csv(lines_path)
    solar_plants = pd.read_csv(solar_path)
else:
    missing = [path.name for path in (substations_path, lines_path, solar_path) if not path.is_file()]
    print(f"Analysis skipped because GIS input files are missing from {GIS}: {', '.join(missing)}")
    print("Fill the corresponding templates under datasets/templates/ and place completed CSVs in the GIS folder.")
    substations = pd.DataFrame(columns=["bus_id", "voltage_kv", "latitude", "longitude"])
    lines = pd.DataFrame(columns=["from_bus", "to_bus", "length_km"])
    solar_plants = pd.DataFrame(columns=["plant_id", "latitude", "longitude"])

substations["bus_id"] = substations["bus_id"].astype(str)
lines["from_bus"] = lines["from_bus"].astype(str)
lines["to_bus"] = lines["to_bus"].astype(str)
if "bus_id" in solar_plants.columns:
    solar_plants["bus_id"] = solar_plants["bus_id"].astype(str)

G = build_grid_graph(substations, lines)
metrics = calculate_graph_metrics(G)
display(metrics.head(10))

             bus_id  degree  ...  betweenness_centrality  closeness_centrality
0          BIYAGAMA       5  ...                0.364860              0.402778
1           KOTMALE       5  ...                0.429475              0.426471
2  NEW_ANURADHAPURA       4  ...                0.266174              0.376623
3          HABARANA       5  ...                0.167529              0.353659
4         VEYANGODA       4  ...                0.140969              0.358025
5      NOROCHCHOLAI       3  ...                0.036289              0.318681
6       PANNIPITIYA       6  ...                0.245115              0.333333
7    KERAWALAPITIYA       2  ...                0.000000              0.311828
8        HAMBANTOTA       3  ...                0.073563              0.268519
9         POLPITIYA       3  ...                0.193924              0.337209

[10 rows x 5 columns]


### Calculate distance to solar plants

In [5]:
metrics = add_solar_distance(
    G,
    metrics,
    solar_plants
)

metrics = metrics.merge(
    substations[["bus_id", "voltage_kv"]],
    on="bus_id",
    how="left"
)

display(metrics.head())

             bus_id  degree  ...  solar_distance_km  voltage_kv
0          BIYAGAMA       5  ...          11.964151         220
1           KOTMALE       5  ...          53.306132         220
2  NEW_ANURADHAPURA       4  ...           0.000000         220
3          HABARANA       5  ...          48.323635         220
4         VEYANGODA       4  ...          36.194154         220

[5 rows x 7 columns]


### Visualize betweenness centrality

In [7]:
if metrics.empty or metrics["betweenness_centrality"].dropna().empty:
    print("Centrality plot skipped: no grid metrics are available. Load completed GIS inputs and rerun the analysis.")
else:
    ax = metrics.sort_values("betweenness_centrality", ascending=False).head(20).plot(
        x="bus_id", y="betweenness_centrality", kind="bar",
        legend=False, figsize=(12, 6)
    )
    ax.set_title("Top 20 Substations by Betweenness Centrality")
    ax.set_ylabel("Betweenness Centrality")
    ax.set_xlabel("Substation")
    ax.figure.tight_layout()
    ax.figure.savefig(GRAPHS / "betweenness_centrality.png", dpi=300)
    plt.show()

### Save Graph Theory results

In [9]:
metrics.to_csv(
    PROCESSED / "graph_theory_metrics.csv",
    index=False
)